In [1]:
import cv2
import os
import csv
import numpy as np
import face_recognition
from datetime import datetime
from collections import defaultdict

class ProfessionalAttendanceSystem:
    """
    Professional Face Recognition Attendance Management System v4.0
    Features real-time face detection, match confidence metrics, multi-frame
    verification, status tracking (On-Time/Late), and live UI notification HUD.
    """
    def __init__(self, students_dir="students", cutoff_time="09:00:00", tolerance=0.55):
        self.students_dir = students_dir
        self.tolerance = tolerance
        try:
            self.cutoff_time = datetime.strptime(cutoff_time, "%H:%M:%S").time()
        except ValueError:
            self.cutoff_time = datetime.strptime("09:00:00", "%H:%M:%S").time()

        self.known_encodings = []
        self.known_names = []
        self.marked_students = {}
        self.detection_counts = defaultdict(int)
        self.notification = None

        self._load_dataset()
        self._init_daily_log()

    def _load_dataset(self):
        """Dynamically scans and encodes student facial datasets."""
        print("==================================================")
        print("   Professional Face Recognition Attendance System ")
        print("==================================================\n")
        supported_exts = ('.jpg', '.jpeg', '.png', '.bmp')
        if not os.path.exists(self.students_dir):
            os.makedirs(self.students_dir)

        files = sorted([f for f in os.listdir(self.students_dir) if f.lower().endswith(supported_exts)])
        print(f"[INFO] Indexing dataset directory '{self.students_dir}'...")
        for filename in files:
            name = os.path.splitext(filename)[0].capitalize()
            img_path = os.path.join(self.students_dir, filename)
            try:
                img = face_recognition.load_image_file(img_path)
                encs = face_recognition.face_encodings(img)
                if encs:
                    self.known_encodings.append(encs[0])
                    self.known_names.append(name)
                    print(f"  [SUCCESS] Face features extracted for: {name}")
                else:
                    print(f"  [WARNING] No face detected in '{filename}'. Skipping.")
            except Exception as e:
                print(f"  [ERROR] Processing '{filename}': {e}")
        print(f"\n[INFO] Dataset ready. Total enrolled students: {len(self.known_names)}\n")

    def _init_daily_log(self):
        """Initializes daily CSV log file and recovers today's existing session records."""
        date_str = datetime.now().strftime("%Y-%m-%d")
        self.log_file = f"{date_str}.csv"
        file_exists = os.path.exists(self.log_file)
        
        if file_exists:
            with open(self.log_file, "r", newline="") as f:
                reader = csv.reader(f)
                header = next(reader, None)
                for row in reader:
                    if len(row) >= 1:
                        name = row[0].strip()
                        t_str = row[1].strip() if len(row) > 1 else ""
                        status = row[3].strip() if len(row) > 3 else "Present"
                        self.marked_students[name] = {"time": t_str, "status": status}
        else:
            with open(self.log_file, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["Name", "Timestamp", "Date", "Status"])

    def mark_attendance(self, name):
        """Records attendance record with On-Time/Late categorization."""
        if name in self.marked_students:
            return False

        now = datetime.now()
        time_str = now.strftime("%H:%M:%S")
        date_str = now.strftime("%Y-%m-%d")
        status = "Late" if now.time() > self.cutoff_time else "On Time"

        with open(self.log_file, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([name, time_str, date_str, status])

        self.marked_students[name] = {"time": time_str, "status": status}
        self.notification = {"message": f"ATTENDANCE MARKED: {name.upper()} ({status})", "time": datetime.now()}
        print(f"[RECORDED] {name} logged as '{status}' at {time_str}")
        return True

    def run(self):
        """Launches main recognition loop with high-resolution visual rendering."""
        video = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        if not video.isOpened():
            video = cv2.VideoCapture(0)

        process_frame = True
        face_locations, face_names, confidences = [], [], []

        try:
            while True:
                ret, frame = video.read()
                if not ret or frame is None:
                    continue

                if process_frame:
                    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
                    rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

                    locations = face_recognition.face_locations(rgb_small)
                    encodings = face_recognition.face_encodings(rgb_small, locations)

                    face_locations = locations
                    face_names = []
                    confidences = []
                    current_detected = set()

                    for encoding in encodings:
                        matches = face_recognition.compare_faces(self.known_encodings, encoding, tolerance=self.tolerance)
                        name = "Unknown"
                        confidence = 0

                        distances = face_recognition.face_distance(self.known_encodings, encoding)
                        if len(distances) > 0:
                            best_idx = np.argmin(distances)
                            best_dist = distances[best_idx]
                            confidence = max(0, min(100, int((1.0 - (best_dist / 1.1)) * 100)))

                            if matches[best_idx]:
                                name = self.known_names[best_idx]
                                current_detected.add(name)
                                self.detection_counts[name] += 1

                                if self.detection_counts[name] >= 2:
                                    self.mark_attendance(name)

                        face_names.append(name)
                        confidences.append(confidence)

                    for k in list(self.detection_counts.keys()):
                        if k not in current_detected:
                            self.detection_counts[k] = max(0, self.detection_counts[k] - 1)

                process_frame = not process_frame

                # Render bounding overlays and UI panels
                frame = self._draw_ui(frame, face_locations, face_names, confidences)

                cv2.imshow("Smart Attendance System v4.0", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print("[INFO] Shutting down attendance service...")
                    break
        finally:
            video.release()
            cv2.destroyAllWindows()

    def _draw_ui(self, frame, face_locations, face_names, confidences):
        """Renders modern HUD elements, bounding boxes, and notification popups."""
        scale = 4
        for (top, right, bottom, left), name, conf in zip(face_locations, face_names, confidences):
            top *= scale
            right *= scale
            bottom *= scale
            left *= scale

            is_known = (name != "Unknown")
            color = (46, 204, 113) if is_known else (52, 73, 94)

            line_len = int(min(right - left, bottom - top) * 0.22)
            thick = 3
            cv2.rectangle(frame, (left, top), (right, bottom), color, 1)
            cv2.line(frame, (left, top), (left + line_len, top), color, thick)
            cv2.line(frame, (left, top), (left, top + line_len), color, thick)
            cv2.line(frame, (right, top), (right - line_len, top), color, thick)
            cv2.line(frame, (right, top), (right, top + line_len), color, thick)
            cv2.line(frame, (left, bottom), (left + line_len, bottom), color, thick)
            cv2.line(frame, (left, bottom), (left, bottom - line_len), color, thick)
            cv2.line(frame, (right, bottom), (right - line_len, bottom), color, thick)
            cv2.line(frame, (right, bottom), (right, bottom - line_len), color, thick)

            status_tag = self.marked_students.get(name, {}).get("status", "")
            label = f"{name} ({conf}%)" if is_known else "Unknown"
            if status_tag:
                label += f" - {status_tag}"

            font = cv2.FONT_HERSHEY_SIMPLEX
            (tw, th), _ = cv2.getTextSize(label, font, 0.55, 1)
            cv2.rectangle(frame, (left, bottom - th - 14), (left + tw + 14, bottom), color, cv2.FILLED)
            cv2.putText(frame, label, (left + 7, bottom - 7), font, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

        h, w, _ = frame.shape
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 50), (20, 24, 33), cv2.FILLED)
        cv2.addWeighted(overlay, 0.85, frame, 0.15, 0, frame)

        total_reg = len(self.known_names)
        present_cnt = len(self.marked_students)
        late_cnt = sum(1 for d in self.marked_students.values() if d["status"] == "Late")
        clock_str = datetime.now().strftime("%H:%M:%S")

        hud_text = f"Enrolled: {total_reg}  |  Present: {present_cnt}  |  Late: {late_cnt}  |  Time: {clock_str}"
        cv2.putText(frame, hud_text, (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (236, 240, 241), 2, cv2.LINE_AA)
        cv2.putText(frame, "[Q] Quit", (w - 95, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (149, 165, 166), 1, cv2.LINE_AA)

        if self.notification:
            elapsed = (datetime.now() - self.notification["time"]).total_seconds()
            if elapsed < 3.5:
                toast_overlay = frame.copy()
                cv2.rectangle(toast_overlay, (20, h - 60), (w - 20, h - 15), (39, 174, 96), cv2.FILLED)
                cv2.addWeighted(toast_overlay, 0.9, frame, 0.1, 0, frame)
                cv2.putText(frame, f"[OK] {self.notification['message']}", (35, h - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                self.notification = None

        return frame

if __name__ == "__main__":
    app = ProfessionalAttendanceSystem(students_dir="students", cutoff_time="09:00:00", tolerance=0.55)
    app.run()
